In [1]:
!pip install trl transformers accelerate peft datasets bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.7 MB/s eta 0:00:00


In [2]:
pip install torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Load ACR guidelines + retrieval (shared with notebook 1)

In [4]:
import re, json

# Same retrieval logic as notebook 1's KD teacher-prompting step, duplicated
# here since this is a separate Kaggle session. Used to build a per-report,
# ACR-augmented system prompt for both SFT training and inference (mod #3).
ACR_GUIDELINES_PATH = "/kaggle/input/datasets/mythreyeehari20/acr-rules-abdomen/Findings_Extracted_WhitePapers.json"

def format_acr_context(guidelines):
    """Condense the ACR abdomen findings into a compact prompt-ready context."""
    
    blocks = []

    for finding in guidelines["findings"]:
        feat_str = "; ".join(finding["features"])

        blocks.append(
            f"- [{finding['finding_id']}] {finding['finding_name']}: "
            f"{finding['description']} "
            f"Key features: {feat_str}"
        )

    return "\n".join(blocks)

with open(ACR_GUIDELINES_PATH) as f:
    acr_guidelines = json.load(f)

ACR_CONTEXT = format_acr_context(acr_guidelines)

def condense_finding(finding):
    """One compact line per finding: name + terse features only, no narrative prose."""
    feat_str = "; ".join(finding["features"])
    return f"[{finding['finding_id']}] {finding['finding_name']} — {feat_str}"


def build_condensed_index(guidelines):
    """Build a searchable index from the abdomen ACR findings."""
    
    index = []

    for finding in guidelines["findings"]:
        searchable = " ".join([
            finding["finding_name"],
            " ".join(finding["features"]),
        ]).lower()

        index.append({
            "finding_id": finding["finding_id"],
            "searchable_text": searchable,
            "condensed_line": condense_finding(finding),
        })

    return index


ACR_INDEX = build_condensed_index(acr_guidelines)

print(f"Indexed {len(ACR_INDEX)} findings")


# Quick size check
full_condensed = "\n".join(
    f["condensed_line"] for f in ACR_INDEX
)

print(
    f"Full condensed size: {len(full_condensed):,} chars "
    f"(vs original ~{len(ACR_CONTEXT):,} chars)"
)


STOPWORDS = {
    "the", "a", "an", "of", "in", "on", "to", "and", "or",
    "with", "is", "are", "was", "were", "at", "for", "by",
    "as", "be", "no", "not", "also", "this", "that",
    "been", "has", "have"
}

def tokenize(text):
    words = re.findall(r"[a-z]+", text.lower())
    return set(w for w in words if w not in STOPWORDS and len(w) > 2)

def retrieve_relevant_findings(report_text, index, top_k=20, min_overlap=1):
    report_tokens = tokenize(report_text)
    scored = []
    for entry in index:
        finding_tokens = tokenize(entry["searchable_text"])
        overlap = len(report_tokens & finding_tokens)
        if overlap >= min_overlap:
            scored.append((overlap, entry))
    scored.sort(key=lambda x: -x[0])
    return [entry for _, entry in scored[:top_k]]

def build_filtered_acr_context(report_text, index, top_k=10):
    relevant = retrieve_relevant_findings(report_text, index, top_k=top_k)
    if not relevant:
        return full_condensed
    return "\n".join(e["condensed_line"] for e in relevant)

TASK_INSTRUCTIONS_TEMPLATE = (
    "You are a clinical assistant specialized in abdominopelvic radiology. Your task is to identify "
    "INCIDENTAL findings in a free-text abdominopelvic CT report — findings unrelated to the report's "
    "primary clinical indication, per ACR Incidental Findings Committee guidelines.\n\n"
    "Use the reference guidelines below to judge whether a finding is clinically incidental "
    "(e.g. small stable nodules, benign-appearing lymph nodes, calcifications) versus a primary/"
    "expected finding tied to the report's main indication.\n\n"
    "Extract the EXACT sentence(s) from the report that describe incidental findings — do not "
    "paraphrase or summarize. If no incidental findings are present, return an empty list.\n\n"
    "REFERENCE GUIDELINES:\n{acr_context}\n\n"
    "OUTPUT FORMAT:\n"
    "Return ONLY one JSON object and nothing else.\n"
    "The JSON object MUST contain exactly these two fields: "
    "contains_IF and incidental_sentences.\n"
    "contains_IF MUST be a boolean: true or false.\n"
    "incidental_sentences MUST be a JSON array of STRINGS, not objects.\n"
    "Each string must be an EXACT sentence copied from the report.\n"
    "Do NOT add fields such as sentence, if, is_incidental, findings, "
    "reference_guidelines, or any other fields.\n"
    "If there are no incidental findings, use an empty array and set contains_IF to false.\n"
    "If incidental findings are present, set contains_IF to true and include only "
    "the exact sentences containing those findings.\n"
    "Required format:\n"
    "{{\"contains_IF\": false, \"incidental_sentences\": []}}\n"
)

def build_system_prompt(report_text, top_k=10):
    filtered_context = build_filtered_acr_context(report_text, ACR_INDEX, top_k=top_k)
    return TASK_INSTRUCTIONS_TEMPLATE.format(acr_context=filtered_context)

Indexed 27 findings
Full condensed size: 13,332 chars (vs original ~16,438 chars)


# Build SFT training data (report -> ACR-augmented prompt -> gold JSON)

In [5]:
import json

# --- Load your pre-split abdomen train/test JSONL files ---
ABDOMEN_TRAIN_PATH = "/kaggle/input/datasets/mythreyee1006/train-dataset-abdomen-new/train_records_abdomenCT.jsonl"  # adjust path
ABDOMEN_TEST_PATH = "/kaggle/input/datasets/mythreyee1006/test-dataset-abdomen-final/test_records_abdomenCT.jsonl"    # adjust path



def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records


train_records_raw = load_jsonl(ABDOMEN_TRAIN_PATH)
test_records_raw = load_jsonl(ABDOMEN_TEST_PATH)
print(f"Train records loaded: {len(train_records_raw)}")
print(f"Test records loaded: {len(test_records_raw)}")


# --- Reconstruct {report_id, free_text, gold} from the messages format ---
def extract_structured(record, idx, prefix):
    messages = record["messages"]
    user_msg = next(m["content"] for m in messages if m["role"] == "user")
    assistant_msg = next(m["content"] for m in messages if m["role"] == "assistant")
    free_text = user_msg.split("Report:\n", 1)[-1]  # strip the "Report:\n" prefix
    gold = json.loads(assistant_msg)  # {"contains_IF": bool, "incidental_sentences": [...]}
    return {
        "report_id": f"{prefix}{idx:04d}",  # synthetic ID, prefixed by split so train/test IDs can't collide
        "free_text": free_text,
        "gold": gold,
    }


train_structured_raw = [extract_structured(r, i, "TRAIN") for i, r in enumerate(train_records_raw)]
test_dataset = [extract_structured(r, i, "TEST") for i, r in enumerate(test_records_raw)]
print(f"Structured train records: {len(train_structured_raw)}")
print(f"Structured test records: {len(test_dataset)}")

# --- Dedup/overlap check ---
# Since IDs are prefixed by split, report_id collisions between train/test are
# impossible by construction -- this check instead catches duplicate FREE TEXT
# across the two files (e.g. the same report accidentally present in both),
# which a report_id-only check would miss entirely here.
test_texts = {r["free_text"].strip() for r in test_dataset}
test_ids = {r["report_id"] for r in test_dataset}

before = len(train_structured_raw)
seen_ids = set()
seen_texts = set()
train_pool = []
dupes_within_train = 0
overlap_with_test = 0

for r in train_structured_raw:
    text_key = r["free_text"].strip()
    if r["report_id"] in test_ids or text_key in test_texts:
        overlap_with_test += 1
        continue
    if r["report_id"] in seen_ids or text_key in seen_texts:
        dupes_within_train += 1
        continue
    seen_ids.add(r["report_id"])
    seen_texts.add(text_key)
    train_pool.append(r)

print(f"Before: {before}")
print(f"Dropped (overlap with test_dataset, by ID or exact text): {overlap_with_test}")
print(f"Dropped (duplicate within train pool, by ID or exact text): {dupes_within_train}")
print(f"Clean train pool: {len(train_pool)}")

if overlap_with_test > 0:
    print("\n⚠️  Overlap found and removed — check whether train/test were meant to be disjoint.")

Train records loaded: 1184
Test records loaded: 100
Structured train records: 1184
Structured test records: 100
Before: 1184
Dropped (overlap with test_dataset, by ID or exact text): 0
Dropped (duplicate within train pool, by ID or exact text): 0
Clean train pool: 1184


# Load merged-KD base model + tokenizer, split data

In [6]:
import torch
import torch.nn.functional as F
import wandb
import numpy as np
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, TrainerCallback, TrainerState, TrainerControl,
    BitsAndBytesConfig,
)
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split

EVAL_BATCH_SIZE = 8
EVAL_STEPS = 50
FOCAL_GAMMA = 2.0

wandb_key = os.environ.get("WANDB_API_KEY")
if wandb_key:
    wandb.login(key=wandb_key)
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("WANDB_API_KEY not set -- running with wandb disabled.")

model_id = "/kaggle/input/datasets/mythreyeehari20/kd-abdomen-weights/kd_merged_abdomen"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# --- Split ONCE, reuse the same split for SFT text and eval ---
train_structured, val_structured = train_test_split(
    train_pool, test_size=100, random_state=42
)
print(f"Train: {len(train_structured)}, Val: {len(val_structured)}")

train_ids = {r["report_id"] for r in train_structured}
val_ids = {r["report_id"] for r in val_structured}


def make_sft_record(record):
    return {
        "report_id": record["report_id"],
        "contains_IF": record["gold"]["contains_IF"],
        "messages": [
            {
                "role": "system",
                "content": build_system_prompt(record["free_text"])
            },
            {
                "role": "user",
                "content": f"Report:\n{record['free_text']}"
            },
            {
                "role": "assistant",
                "content": json.dumps(
                    {
                        "contains_IF": record["gold"]["contains_IF"],
                        "incidental_sentences": record["gold"]["incidental_sentences"]
                    },
                    ensure_ascii=False
                )
            }
        ]
    }


train_records = [
    make_sft_record(r)
    for r in train_structured
]

val_records_chat = [
    make_sft_record(r)
    for r in val_structured
]

print(f"SFT train records: {len(train_records)}")
print(f"SFT validation records: {len(val_records_chat)}")

# --- Class weights, computed from the actual train split ---
n_pos = sum(1 for r in train_records if r["contains_IF"])
n_neg = len(train_records) - n_pos
n_total = len(train_records)
class_weight = {
    True:  n_total / (2 * n_pos),
    False: n_total / (2 * n_neg),
}
print(f"n_pos={n_pos}, n_neg={n_neg}, class_weight={class_weight}")

# --- Tokenize with left-truncation (never clips the gold completion) ---
MAX_LEN = 3072

def tokenize_with_labels(example):
    system_msg, user_msg, assistant_msg = example["messages"]
    prompt_text = tokenizer.apply_chat_template(
        [system_msg, user_msg], tokenize=False, add_generation_prompt=True
    )
    full_text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]

    if len(full_ids) > MAX_LEN:
        overflow = len(full_ids) - MAX_LEN
        full_ids = full_ids[overflow:]
        prompt_len = max(len(prompt_ids) - overflow, 0)
    else:
        prompt_len = len(prompt_ids)

    labels = [-100] * prompt_len + full_ids[prompt_len:]
    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
        "class_weight": class_weight[example["contains_IF"]],
    }

train_dataset = Dataset.from_list(train_records).map(
    tokenize_with_labels,
    remove_columns=["messages", "report_id", "contains_IF"],
)

token_lengths = [len(ex) for ex in train_dataset["input_ids"]]
print(f"Token lengths -- min: {min(token_lengths)}, max: {max(token_lengths)}, "
      f"mean: {np.mean(token_lengths):.0f}, p95: {np.percentile(token_lengths, 95):.0f}")

WANDB_API_KEY not set -- running with wandb disabled.
Train: 1084, Val: 100
SFT train records: 1084
SFT validation records: 100
n_pos=424, n_neg=660, class_weight={True: 1.278301886792453, False: 0.8212121212121212}


Map:   0%|          | 0/1084 [00:00<?, ? examples/s]

Token lengths -- min: 435, max: 3072, mean: 1732, p95: 2211


# Eval helpers (fuzzy matching) + early-stopping callback

In [7]:
import re
from difflib import SequenceMatcher


def parse_output(raw_text):
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{.*\}", raw_text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return None


def build_messages(record, few_shot_pool=None, n_shot=0):
    """mod #3: ACR-augmented system prompt is built per-record instead of a
    single flat SYSTEM_PROMPT."""
    messages = [{"role": "system", "content": build_system_prompt(record["free_text"])}]

    if few_shot_pool and n_shot > 0:
        for ex in few_shot_pool[:n_shot]:
            messages.append({"role": "user", "content": f"Report:\n{ex['free_text']}"})
            messages.append({
                "role": "assistant",
                "content": json.dumps({
                    "contains_IF": ex["gold"]["contains_IF"],
                    "incidental_sentences": ex["gold"]["incidental_sentences"],
                }),
            })

    messages.append({"role": "user", "content": f"Report:\n{record['free_text']}"})
    return messages


# --- mod #1: fuzzy matching eval, replaces exact-set-overlap scoring ---
def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def fuzzy_match_sets(gold_set, pred_set, threshold=0.85):
    """Greedy one-to-one fuzzy matching between gold and predicted sentences."""
    gold_list = list(gold_set)
    pred_list = list(pred_set)
    matched_gold, matched_pred = set(), set()
    pairs = []
    for gi, g in enumerate(gold_list):
        for pi, p in enumerate(pred_list):
            sim = similarity(g, p)
            if sim >= threshold:
                pairs.append((sim, gi, pi))
    pairs.sort(key=lambda x: -x[0])
    for sim, gi, pi in pairs:
        if gi in matched_gold or pi in matched_pred:
            continue
        matched_gold.add(gi)
        matched_pred.add(pi)
    tp = len(matched_gold)
    fp = len(pred_list) - len(matched_pred)
    fn = len(gold_list) - len(matched_gold)
    return tp, fp, fn


def run_evaluation_fuzzy(results, label="", threshold=0.85):
    parse_failures = sum(r["parse_failed"] for r in results)
    valid = [r for r in results if not r["parse_failed"]]
    total_tp = total_fp = total_fn = 0
    neg_scores, pos_scores = [], []
    for r in valid:
        gold_set = set(s.strip().lower() for s in r["gold_sentences"])
        pred_set = set(s.strip().lower() for s in (r["pred_sentences"] or []))
        if len(gold_set) == 0:
            neg_scores.append(1.0 if len(pred_set) == 0 else 0.0)
        else:
            tp, fp, fn = fuzzy_match_sets(gold_set, pred_set, threshold=threshold)
            total_tp += tp; total_fp += fp; total_fn += fn
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
            pos_scores.append(f1)
    avg_neg = sum(neg_scores) / len(neg_scores) if neg_scores else 0.0
    avg_pos = sum(pos_scores) / len(pos_scores) if pos_scores else 0.0
    n_neg, n_pos = len(neg_scores), len(pos_scores)
    if n_neg > 0 and n_pos > 0:
        macro_f1 = (avg_neg + avg_pos) / 2
    elif n_neg > 0:
        macro_f1 = avg_neg
    elif n_pos > 0:
        macro_f1 = avg_pos
    else:
        macro_f1 = 0.0
    weighted_f1 = (n_neg * avg_neg + n_pos * avg_pos) / (n_neg + n_pos) if (n_neg + n_pos) > 0 else 0.0
    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    micro_f1 = (2 * micro_precision * micro_recall / (micro_precision + micro_recall)
                if (micro_precision + micro_recall) > 0 else 0.0)
    print(f"\n{'='*50}\n=== {label} (fuzzy threshold={threshold}) ===\n{'='*50}")
    print(f"Parse failures:      {parse_failures}/{len(results)}")
    print(f"Negative-report acc: {avg_neg:.4f}  (n={n_neg})")
    print(f"Positive-report F1:  {avg_pos:.4f}  (n={n_pos})")
    print(f"Sentence Macro F1:   {macro_f1:.4f}")
    print(f"Sentence Weighted:   {weighted_f1:.4f}")
    print(f"Sentence Micro F1:   {micro_f1:.4f}")
    return {"macro_f1": macro_f1, "weighted_f1": weighted_f1, "micro_f1": micro_f1,
            "parse_failures": parse_failures, "n_neg": n_neg, "n_pos": n_pos,
            "avg_neg": avg_neg, "avg_pos": avg_pos}


def generate_predictions(model, tokenizer, records, few_shot_pool=None, n_shot=0,
                          n=None, batch_size=EVAL_BATCH_SIZE):
    import transformers
    transformers.logging.set_verbosity_error()
    model.eval()

    subset = records[:n] if n else records
    results = []

    for start in range(0, len(subset), batch_size):
        batch_records = subset[start:start + batch_size]
        texts = [
            tokenizer.apply_chat_template(
                build_messages(r, few_shot_pool, n_shot),
                tokenize=False, add_generation_prompt=True
            )
            for r in batch_records
        ]
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[-1]
        for i, record in enumerate(batch_records):
            generated = output_ids[i][prompt_len:]
            raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
            parsed = parse_output(raw)

            results.append({
                "report_id": record["report_id"],
                "gold_sentences": record["gold"]["incidental_sentences"],
                "pred_sentences": parsed.get("incidental_sentences", []) if parsed else None,
                "parse_failed": parsed is None,
            })

    return results


class MacroF1EarlyStoppingCallback(TrainerCallback):
    def __init__(self, eval_records, tokenizer, run_name, eval_steps=EVAL_STEPS, patience=3):
        self.eval_records = eval_records
        self.tokenizer = tokenizer
        self.run_name = run_name
        self.eval_steps = eval_steps
        self.patience = patience
        self.best_f1 = -1.0
        self.no_improve = 0
        self.best_step = 0
        self.best_metrics = None
        self.best_ckpt_dir = f"./qlora_best_{run_name}"

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.eval_steps != 0 or state.global_step == 0:
            return control
    
        model = kwargs["model"]
        results = generate_predictions(model, self.tokenizer, self.eval_records, n=100)
        metrics = run_evaluation_fuzzy(results, label=f"Step {state.global_step}")
    
        wandb.log({
            "val_macro_f1": metrics["macro_f1"],
            "val_micro_f1": metrics["micro_f1"],
            "val_weighted_f1": metrics["weighted_f1"],
            "val_avg_neg": metrics["avg_neg"],
            "val_avg_pos": metrics["avg_pos"],
            "val_parse_failures": metrics["parse_failures"],
            "step": state.global_step,
        })
    
        if metrics["macro_f1"] > self.best_f1:
            self.best_f1 = metrics["macro_f1"]
            self.best_step = state.global_step
            self.best_metrics = metrics
            self.no_improve = 0
            model.save_pretrained(self.best_ckpt_dir)
            self.tokenizer.save_pretrained(self.best_ckpt_dir)
            print(f"  New best saved -> {self.best_ckpt_dir}")
        else:
            self.no_improve += 1
            print(f"  No improvement ({self.no_improve}/{self.patience})")
    
        if self.no_improve >= self.patience:
            print(f"\nEarly stopping at step {state.global_step}. Best step {self.best_step}, macro F1={self.best_f1:.4f}")
            control.should_training_stop = True
    
        return control

# QLoRA training with focal loss, on top of the merged KD checkpoint

In [8]:
class WeightedDataCollator:
    def __init__(self, tokenizer):
        self.pad_token_id = tokenizer.pad_token_id

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        input_ids, attention_mask, labels, weights = [], [], [], []
        for f in features:
            pad_n = max_len - len(f["input_ids"])
            input_ids.append(f["input_ids"] + [self.pad_token_id] * pad_n)
            attention_mask.append(f["attention_mask"] + [0] * pad_n)
            labels.append(f["labels"] + [-100] * pad_n)
            weights.append(f["class_weight"])
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "class_weight": torch.tensor(weights, dtype=torch.float32),
        }

In [9]:
import gc

# Hyperparameter grid -- unchanged from your original run
lora_configs = [
    {"r": 32, "lora_alpha": 64},
]

sweep_results = []

# 4-bit NF4 quantization config.
# Changed from FP16 compute to BF16 to test/fix the NaN logits
# observed with the FP16 quantized forward pass.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


class WeightedCETrainer(SFTTrainer):
    """Plain CE with a per-example class weight."""
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs.pop("labels")
        weights = inputs.pop("class_weight")

        outputs = model(**inputs)
        logits = outputs.logits

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        log_probs = F.log_softmax(shift_logits.float(), dim=-1)

        B, L, V = log_probs.shape

        ce = F.nll_loss(
            log_probs.view(-1, V),
            shift_labels.view(-1),
            ignore_index=-100,
            reduction="none",
        ).view(B, L)

        valid_mask = (shift_labels != -100).float()

        per_example_weight = (
            weights
            .to(ce.device, ce.dtype)
            .unsqueeze(1)
        )

        loss = (
            (ce * valid_mask * per_example_weight).sum()
            /
            (valid_mask * per_example_weight)
            .sum()
            .clamp(min=1e-8)
        )

        return (loss, outputs) if return_outputs else loss


class FocalLossSFTTrainer(SFTTrainer):
    """Focal loss with class-weighted alpha.
    Kept as a fallback; not used in this run.
    """
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs.pop("labels")
        weights = inputs.pop("class_weight")

        outputs = model(**inputs)
        logits = outputs.logits

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        log_probs = F.log_softmax(shift_logits.float(), dim=-1)

        B, L, V = log_probs.shape

        ce = F.nll_loss(
            log_probs.view(-1, V),
            shift_labels.view(-1),
            ignore_index=-100,
            reduction="none",
        ).view(B, L)

        valid_mask = (shift_labels != -100).float()

        with torch.no_grad():
            pt = (
                log_probs.view(-1, V)
                .gather(
                    1,
                    shift_labels.view(-1)
                    .clamp(min=0)
                    .unsqueeze(1),
                )
                .squeeze(1)
                .exp()
                .view(B, L)
            )

        focal_term = (1 - pt).pow(FOCAL_GAMMA)

        per_example_weight = (
            weights
            .to(ce.device, ce.dtype)
            .unsqueeze(1)
        )

        loss = (
            (focal_term * ce * valid_mask * per_example_weight).sum()
            /
            (valid_mask * per_example_weight)
            .sum()
            .clamp(min=1e-8)
        )

        return (loss, outputs) if return_outputs else loss


for config in lora_configs:
    run_name = f"r{config['r']}_alpha{config['lora_alpha']}"

    print(f"\n{'=' * 50}")
    print(f"Starting run: {run_name}")
    print(f"{'=' * 50}")

    wandb.init(
        project="incidental-findings-finetuning",
        name=f"kd-then-qlora-{run_name}",
        config=config,
        reinit=True,
    )

    # Load the merged KD checkpoint.
    # This is QLoRA on top of KD, not QLoRA from the original Qwen model.
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        torch_dtype=torch.bfloat16,
        device_map={"": 0},
    )

    model = prepare_model_for_kbit_training(model)

    print("Model config torch_dtype:", model.config.torch_dtype)

    for name, param in model.named_parameters():
        if param.requires_grad:
            print(
                "First trainable parameter:",
                name,
                param.dtype,
            )
            break

    peft_config = LoraConfig(
        r=config["r"],
        lora_alpha=config["lora_alpha"],
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
        ],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    training_args = SFTConfig(
        output_dir=f"./qwen_lora_{run_name}",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        max_length=3072,
        learning_rate=2e-4,
        logging_steps=10,
        num_train_epochs=100,
        save_strategy="steps",
        save_steps=50,
        eval_strategy="no",
        lr_scheduler_type="cosine",
        warmup_steps=10,
        fp16=False,
        bf16=True,
        report_to="wandb",
        remove_unused_columns=False,
        dataset_kwargs={
            "skip_prepare_dataset": True
        },
        gradient_checkpointing=True,
    )

    callback = MacroF1EarlyStoppingCallback(
        eval_records=val_structured,
        tokenizer=tokenizer,
        run_name=run_name,
        eval_steps=50,
        patience=3,
    )

    trainer = WeightedCETrainer(
        model=model,
        train_dataset=train_dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
        args=training_args,
        callbacks=[callback],
        data_collator=WeightedDataCollator(tokenizer),
    )

    print("Training examples:", len(train_dataset))

    # Keep LoRA parameters in BF16 for this BF16 test.
    # Do NOT convert them to FP16 here.

    dtype_counts = {}

    for name, param in trainer.model.named_parameters():
        if param.requires_grad:
            dtype_counts.setdefault(
                param.dtype,
                []
            ).append(name)

    print("\nTrainable parameter dtypes:")

    for dtype, names in dtype_counts.items():
        print(
            f"{dtype}: "
            f"{len(names)} trainable params, "
            f"e.g. {names[0]}"
        )

    # ---------------------------------------------------------
    # Verify the custom batch/collator
    # ---------------------------------------------------------

    batch = next(iter(trainer.get_train_dataloader()))

    print("\nBatch keys:", batch.keys())

    assert "class_weight" in batch, \
        "class_weight was dropped again!"

    print(
        "class_weight sample:",
        batch["class_weight"][:5]
    )

    # ---------------------------------------------------------
    # IMPORTANT:
    # Test the forward pass BEFORE training.
    # The previous FP16 setup produced NaN logits.
    # ---------------------------------------------------------

    print("\nChecking forward pass before training...")

    with torch.no_grad():
        outputs = trainer.model(
            input_ids=batch["input_ids"].to(model.device),
            attention_mask=batch["attention_mask"].to(model.device),
        )

    logits = outputs.logits

    print("Logits dtype:", logits.dtype)
    print(
        "Logits NaN:",
        torch.isnan(logits).sum().item()
    )
    print(
        "Logits Inf:",
        torch.isinf(logits).sum().item()
    )

    if torch.isnan(logits).any():
        raise RuntimeError(
            "NaN logits detected before training. "
            "BF16 did not resolve the numerical instability."
        )

    if torch.isinf(logits).any():
        raise RuntimeError(
            "Inf logits detected before training."
        )

    print("Forward pass is numerically stable.")

    # Free the temporary forward-pass outputs before training.
    del outputs, logits
    gc.collect()
    torch.cuda.empty_cache()

    # ---------------------------------------------------------
    # Start training
    # ---------------------------------------------------------

    trainer.train()

    # ---------------------------------------------------------
    # Store best result
    # ---------------------------------------------------------

    sweep_results.append({
        "run": run_name,
        "r": config["r"],
        "lora_alpha": config["lora_alpha"],
        "best_f1": callback.best_f1,
        "best_step": callback.best_step,
    })

    wandb.finish()

    # ---------------------------------------------------------
    # Free GPU memory before next sweep iteration
    # ---------------------------------------------------------

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()


# -------------------------------------------------------------
# Summary
# -------------------------------------------------------------

print("\n" + "=" * 50)
print("Sweep Summary:")
print("=" * 50)

for r in sorted(
    sweep_results,
    key=lambda x: x["best_f1"],
    reverse=True,
):
    print(
        f"  {r['run']:20s} | "
        f"Best F1: {r['best_f1']:.4f} | "
        f"Step: {r['best_step']}"
    )

`torch_dtype` is deprecated! Use `dtype` instead!



Starting run: r32_alpha64


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model config torch_dtype: torch.bfloat16
Training examples: 1084

Trainable parameter dtypes:
torch.bfloat16: 192 trainable params, e.g. base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight

Batch keys: dict_keys(['input_ids', 'attention_mask', 'labels', 'class_weight'])
class_weight sample: tensor([1.2783], device='cuda:0')

Checking forward pass before training...
Logits dtype: torch.float32
Logits NaN: 0
Logits Inf: 0
Forward pass is numerically stable.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss
10,0.567419
20,0.487468
30,2.257093
40,0.539061
50,0.601243
60,0.426861
70,0.383185
80,0.342112
90,0.380772
100,0.906809



=== Step 50 (fuzzy threshold=0.85) ===
Parse failures:      1/100
Negative-report acc: 0.9538  (n=65)
Positive-report F1:  0.2843  (n=34)
Sentence Macro F1:   0.6191
Sentence Weighted:   0.7239
Sentence Micro F1:   0.3056
  New best saved -> ./qlora_best_r32_alpha64

=== Step 100 (fuzzy threshold=0.85) ===
Parse failures:      1/100
Negative-report acc: 0.8281  (n=64)
Positive-report F1:  0.3171  (n=35)
Sentence Macro F1:   0.5726
Sentence Weighted:   0.6475
Sentence Micro F1:   0.3556
  No improvement (1/3)

=== Step 150 (fuzzy threshold=0.85) ===
Parse failures:      0/100
Negative-report acc: 0.7692  (n=65)
Positive-report F1:  0.4305  (n=35)
Sentence Macro F1:   0.5999
Sentence Weighted:   0.6507
Sentence Micro F1:   0.4186
  No improvement (2/3)

=== Step 200 (fuzzy threshold=0.85) ===
Parse failures:      1/100
Negative-report acc: 0.9062  (n=64)
Positive-report F1:  0.4676  (n=35)
Sentence Macro F1:   0.6869
Sentence Weighted:   0.7512
Sentence Micro F1:   0.4468
  New best sav

# Standalone evaluation (run after training)

In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ============================================================
# Paths
# ============================================================

kd_model_path = "/kaggle/input/datasets/mythreyeehari20/kd-abdomen-weights/kd_merged_abdomen"

# Change this to the checkpoint selected as BEST by your
# early-stopping callback.
qlora_checkpoint = "./qlora_best_r32_alpha64"   # the callback's tracked best, not a Trainer step checkpoint

output_path = "./abdomen_kd_qlora_merged"


# ============================================================
# Load tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    kd_model_path,
    trust_remote_code=True
)


# ============================================================
# Load KD model in normal precision
#
# IMPORTANT:
# For merging, don't load the base model in 4-bit.
# ============================================================

base_model = AutoModelForCausalLM.from_pretrained(
    kd_model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)


# ============================================================
# Load QLoRA adapter
# ============================================================

model = PeftModel.from_pretrained(
    base_model,
    qlora_checkpoint
)


# ============================================================
# Merge LoRA weights into KD model
# ============================================================

print("Merging QLoRA adapter into KD model...")

merged_model = model.merge_and_unload()

print("Merge complete.")


# ============================================================
# Save final merged model
# ============================================================

merged_model.save_pretrained(
    output_path,
    safe_serialization=True
)

tokenizer.save_pretrained(output_path)


print()
print("==============================================")
print("Merge Successful!")
print("==============================================")
print(f"Saved final model to: {output_path}")
print("==============================================")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Merging QLoRA adapter into KD model...
Merge complete.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Merge Successful!
Saved final model to: ./abdomen_kd_qlora_merged


In [11]:
# --- Final evaluation on the real held-out test set ---
merged_model.eval()
test_results = generate_predictions(merged_model, tokenizer, test_dataset, n=None, batch_size=2)
final_metrics = run_evaluation_fuzzy(test_results, label="KD+QLoRA merged -- abdomen test set")


=== KD+QLoRA merged -- abdomen test set (fuzzy threshold=0.85) ===
Parse failures:      12/100
Negative-report acc: 0.9423  (n=52)
Positive-report F1:  0.5066  (n=36)
Sentence Macro F1:   0.7245
Sentence Weighted:   0.7641
Sentence Micro F1:   0.5349
